In [ ]:
import sys
sys.path.append("..")

import numpy as np
from numpy.linalg import norm
from scipy.linalg import expm, block_diag

import matplotlib.pyplot as plt
from lunanav.constants import GM_MOON, R_MOON, RAD_TO_DEG
from lunanav.sim.simulator import SimParams, SimResults, run_sim, RigidBody, calc_measurements, SensorNoises
from lunanav.sim.sensors import get_los_vectors, dist_from_los, dist_rate_from_los, meas_range_tracker
from lunanav.plotting import debug_3d, plot_state_vector, plot_control_effort
from lunanav.visualization import visualize_trajectory
from lunanav.sim.quaternion import angle_axis_to_q, mul, quat_apply, unit, unitize_state
from lunanav.estimation.ekf import EkfParams, ekf_predict, ekf_update, Qd_from_accel_white
import jax
import jax.numpy as jnp

# Sim setup (no doppler yet)

In [ ]:
@jax.jit
def control_fn(t, state):
    """Outputs inputs in body frame"""
    del state
    force_N = jnp.zeros(3)
    torque_Nm = jnp.zeros(3)

    # ----- Thrust (body z) -----
    fz = jnp.where(t < 5,  300.0,
         jnp.where(t < 25, 400.0,
         jnp.where(t < 80, 1500.0,
         jnp.where(t < 200, 300.0, 0.0))))
    force_N = jnp.array([0.0, 0.0, fz])

    # ----- Torque (body x) -----
    tx = jnp.where(t < 6,   0.007,
         jnp.where(t < 18, -0.005,
         jnp.where(t < 36,  0.003,
         jnp.where(t < 100, -0.001, 0.0))))
    torque_Nm = jnp.array([tx, 0.0, 0.0])

    return force_N, torque_Nm


## Simulate

In [ ]:
lander = RigidBody(
    mass_kg = 100,
    I = np.eye(3)
)

dt = .1
t0 = 0
t_max = 100
nsteps = int(t_max//dt)

state0 = np.array([
    0,0, R_MOON + 3, 
    0 ,0, 0,
    1,0,0,0, 
    0,0,0])

In [ ]:
sim = SimParams(state0, lander, dt, nsteps)
results = run_sim(state0, nsteps, dt, control_fn, sim)
n = results.nsteps

In [ ]:
moon_offset =  np.tile([0,0,R_MOON,0,0,0,0,0,0,0,0,0,0], (n, 1))

other_vecs = {
    "names": ["LOS1", "LOS2", "LOS3", "LOS4"],
    "vecs": get_los_vectors(),
    "colors": ['green', 'green', 'green', 'green'],
    "scale": 1e3
}
fig = visualize_trajectory(results.states - moon_offset, results.t, dt, title="Lunar Descent Trajectory with LOS Vectors", show_lander=True, other_vecs=other_vecs, downsample_rate=100)
fig.show()

In [ ]:
plot_state_vector(results.t, results.states[:,0:3], results.states[:,3:6], results.states[:,10:13], figsize=(16,10))

## Measurements

In [ ]:
range_tracker_pos = [state0[0:6],
                  state0[0:6] + [10e3, -15e3, 0, 0, 0, 0],
                  state0[0:6] + [-10e3, -15e3, 0, 0, 0, 0]
                  ]

In [ ]:
sigma_accel = 0.1  # m/s^2
sigma_gyro = 1e-3  # rad/s^2
sigma_los = 100  # m
sigma_los_vel = 10  # m/s
sigma_star_tracker = 1e-2  # unitless

R_accel = np.eye(3)*sigma_accel**2
R_gyro = np.eye(3)*sigma_gyro**2
R_los = np.eye(4)*sigma_los**2
R_los_vel = np.eye(4)*sigma_los_vel**2
R_star_tracker = np.eye(4)*sigma_star_tracker**2
R_range_tracker = np.diag(np.tile([sigma_los**2]*3 + [sigma_los_vel**2]*3, 3))

truth_meas = calc_measurements(results, lander.mass_kg, range_tracker_pos=range_tracker_pos)
noise_meas = calc_measurements(results, lander.mass_kg, SensorNoises(R_accel, R_gyro, R_los, R_los_vel, R_star_tracker, R_range_tracker), range_tracker_pos=range_tracker_pos)

In [ ]:
plt.plot(results.t[:-1], noise_meas.accel[:-1])
plt.xlabel("Time (s)")
plt.ylabel("Measured Acceleration (m/s^2)")
plt.title("Measured Acceleration vs Time")
plt.grid()
plt.show()

plt.plot(results.t[:-1], noise_meas.gyro[:-1] * RAD_TO_DEG)
plt.xlabel("Time (s)")
plt.ylabel("Measured Angular Velocity (deg/s)")
plt.title("Measured Angular Velocity vs Time")
plt.grid()
plt.show()

plt.plot(results.t[:-1], noise_meas.laser_alt[:-1])
plt.xlabel("Time (s)")
plt.ylabel("Distance (m)")
plt.title("Measured Laser Distance vs Time")
plt.grid()
plt.show()

plt.plot(results.t[:-1], noise_meas.laser_vel[:-1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("Measured Laser Velocity vs Time")
plt.grid()
plt.show()

plt.plot(results.t[:-1], noise_meas.star_tracker[:-1])
plt.xlabel("Time (s)")
# plt.ylabel("Measured Laser Velocity (m)")
plt.title("Measured Star Tracker Quat Elements vs Time")
plt.grid()
plt.show()

# plt.plot(results.t[:-1], noise_meas.range_tracker[:-1])
# plt.xlabel("Time (s)")
# # plt.ylabel("Measured Laser Velocity (m)")
# plt.title("Measured Range Tracker Elements vs Time")
# plt.grid()
# plt.show()

## Ekf

In [ ]:
def ekf_alt_func(state):
    """Measurement function for EKF"""
    meas_arr = dist_from_los(state)
    return meas_arr

def ekf_los_vel_func(state):
    """Measurement function for EKF"""
    meas_arr = dist_rate_from_los(state)
    return meas_arr

def ekf_star_tracker_func(state):
    """Measurement function for EKF"""
    q_B2L = state[6:10]
    return q_B2L

# def ekf_range_tracker_func(state):
#     r_lander = state[0:3]
#     v_lander = state[3:6]
    
#     # Stack all [r_rel, v_rel] pairs
#     measurements = []
#     for s in range_tracker_pos:
#         rel_pos = r_lander - s[0:3]
#         rel_vel = v_lander - s[3:6]  # launch sites might be moving; subtract their vel
#         measurements.append(jnp.concatenate([rel_pos, rel_vel]))
    
#     return jnp.concatenate(measurements)

# Compute the Jacobian (JIT SAVES SO MUCH TIME)
ekf_alt_jacobian = jax.jit(jax.jacfwd(ekf_alt_func))
ekf_los_vel_jacobian = jax.jit(jax.jacfwd(ekf_los_vel_func))
ekf_star_tracker_jacobian = jax.jit(jax.jacfwd(ekf_star_tracker_func))
# ekf_range_tracker_jacobian = jax.jit(jax.jacfwd(ekf_range_tracker_func))

In [ ]:
R_accel_ekf = R_accel
R_gyro_ekf = R_gyro
R_los_ekf = R_los
R_los_vel_ekf = R_los_vel
R_star_tracker_ekf = R_star_tracker
R_range_tracker_ekf = R_range_tracker

Q_ekf = np.zeros((13,13))
Q_ekf[0:6, 0:6] = Qd_from_accel_white(dt, sigma_accel)

# Gyro noise → quaternion/angular velocity uncertainty  
Q_ekf[6:10, 6:10] = np.eye(4) * 1e-6  # quaternion
Q_ekf[10:13, 10:13] = np.eye(3) * sigma_gyro**2 * dt  # angular velocity

# Q_ekf *= 0.1 

In [ ]:
mu_arr = np.zeros((n, 13))
Sigma_arr = np.zeros((n, 13, 13))

mu_arr[0] = state0
Sigma_arr[0] = np.eye(13) * 1e-3

n_new = n

for i in range(n_new-1):

    
    # force, torque = control_fn(results.t[i], results.states[i])
    mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], noise_meas.accel[i], noise_meas.gyro[i], Q_ekf, sim)
    mu_pred = unitize_state(mu_pred)

    # if results.t[i] < 30:

    if i % 10 and jnp.sum(noise_meas.laser_alt[i]) < 1e6:
        mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, noise_meas.laser_alt[i], ekf_alt_jacobian(mu_pred), ekf_alt_func(mu_pred), R_los_ekf)
        mu_pred = unitize_state(mu_pred)
        mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, noise_meas.laser_vel[i], ekf_los_vel_jacobian(mu_pred), ekf_los_vel_func(mu_pred), R_los_vel_ekf)
        mu_pred = unitize_state(mu_pred)
        # mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, noise_meas.range_tracker[i], ekf_range_tracker_jacobian(mu_pred), ekf_range_tracker_func(mu_pred), R_range_tracker_ekf)
        # mu_pred = unitize_state(mu_pred)


    if i % 50 == 0:
        mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, noise_meas.star_tracker[i], ekf_star_tracker_jacobian(mu_pred), ekf_star_tracker_func(mu_pred), R_star_tracker_ekf)
        mu_pred = unitize_state(mu_pred)

    # z_combined = np.concatenate([noise_meas.laser_alt[i], noise_meas.laser_vel[i]])
    # h_combined = np.concatenate([ekf_alt_func(mu_pred), ekf_los_vel_func(mu_pred)])
    # H_combined = np.vstack([ekf_alt_jacobian(mu_pred), ekf_los_vel_jacobian(mu_pred)])
    # R_combined = np.block([[R_los_ekf, np.zeros((4,4))],
    #                     [np.zeros((4,4)), R_los_vel_ekf]])

    # mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, z_combined, H_combined, h_combined, R_combined)
    # mu_pred = unitize_state(mu_pred)

    mu_arr[i+1] = mu_pred
    Sigma_arr[i+1] = Sigma_pred

    if i % 50 == 0:
        print(f"{i} of {n}")
    #     truth = results.states[i]
    #     err = mu_pred - truth
    #     sigma_diag = np.sqrt(np.diag(Sigma_pred))
        
    #     # Position
    #     p_err_norm = np.linalg.norm(err[0:3])
    #     p_sigma_norm = np.linalg.norm(sigma_diag[0:3])
    #     print(f"t={results.t[i]:6.1f}: |p_err|={p_err_norm:.2f} m, "
    #         f"|p_σ|={p_sigma_norm:.2f} m, ratio={p_err_norm/p_sigma_norm:.2f}")
        
    #     # Velocity
    #     v_err_norm = np.linalg.norm(err[3:6])
    #     v_sigma_norm = np.linalg.norm(sigma_diag[3:6])
    #     print(f"        |v_err|={v_err_norm:.3f} m/s, "
    #         f"|v_σ|={v_sigma_norm:.3f} m/s, ratio={v_err_norm/v_sigma_norm:.2f}")

    # mu_arr[i+1] = mu_update
    # Sigma_arr[i+1] = Sigma_update

    t = results.t[i+1]

    
    if False:

        x_est = mu_pred
        noise = results.states[i+1]
        a_meas = noise_meas.accel[i]
        w_meas = noise_meas.gyro[i]
        P = Sigma_pred
        alt_meas = noise_meas.laser_alt[i]
        
        print(f"t={t:6.2f}: "
            f"||q||={np.linalg.norm(x_est[6:10]):.8f} "
            f"q_err={np.linalg.norm(noise[6:10] - x_est[6:10]):.6f} "
            f"v_err={np.linalg.norm(noise[3:6] - x_est[3:6]):.4f} "
            f"p_err={np.linalg.norm(noise[0:3] - x_est[0:3]):.4f} "
            f"||P_qq||={np.linalg.norm(P[6:10, 6:10]):.2e} "
            f"||a_meas||={np.linalg.norm(a_meas):.4f} "
            f"||w_meas||={np.linalg.norm(w_meas):.6f} "
            f"Alts: {alt_meas}"
            )
    # z = results.states[i+1][0:6]  # True next state (pos and vel)
    # H = np.hstack([np.eye(6), np.zeros((6,4))])  # Measurement matrix to extract pos and vel
    # mu0, Sigma0 = ekf_update(mu_pred, Sigma_pred, z, H, R)

In [ ]:
visualize_trajectory([mu_arr - moon_offset, results.states - moon_offset], results.t, dt, title="EKF Estimated Trajectory with LOS Vectors", show_lander=False, downsample_rate=100).show()

In [ ]:
plot_state_vector(results.t, mu_arr[:,0:3], mu_arr[:,3:6], mu_arr[:,10:13])

In [ ]:
plot_state_vector(results.t, results.states[:,0:3] - mu_arr[:,0:3], results.states[:,3:6] - mu_arr[:,3:6], results.states[:,10:13] - mu_arr[:,10:13])

In [ ]:
plt.figure(figsize=(10,6))
errs = np.linalg.norm(mu_arr[:,:3] - results.states[:,:3], axis = 1)
vel_errs = np.linalg.norm(mu_arr[:,3:6] - results.states[:,3:6], axis = 1)

# plt.plot(results.t, vel_errs)
plt.plot(results.t, errs)
plt.xlabel("Time (s)")
plt.ylabel("Z-position Error (m)")
plt.title("EKF Position Estimation Error vs Time")
plt.grid()
# plt.legend()
plt.show()

# With doppler